# 02 · Datos — descarga de FaceForensics++ (c23) y recorte de caras
Genera los recortes de cara (299×299, 16 frames por video) **solo** para los videos que usan los experimentos y los entrega como un único archivo para crear el *Kaggle Dataset* privado `ffpp-crops-g8` (D-05, D-12).

| Split | Pares oficiales usados | Categorías |
|---|---|---|
| train | 100 de 360 | original + Deepfakes |
| val | 30 de 70 | original + Deepfakes |
| test | 70 de 70 | original + Deepfakes, Face2Face, FaceSwap, NeuralTextures |

**Output de este notebook (solo 3 archivos y la carpeta `privado/`):** `crops.tar` (~3 GB), `manifest.csv` (recortes y descartes por video) y `crops_meta.json`.

**Cómo usarlo**
1. Settings: Accelerator = GPU T4 · Internet = On · Input: el Dataset privado con el script oficial de descarga.
2. **Save Version → Save & Run All (Commit)**. El output solo se puede convertir en Dataset desde una versión guardada, no desde la sesión interactiva.
3. En la versión terminada: pestaña *Output* → **New Dataset** → nombre `ffpp-crops-g8`, visibilidad **Private**.
4. El 03 (y la sección 5 del 01) lo agregan como input y lo descomprimen solos.

> **Licencia FF++ y sincronización con GitHub.** Kaggle sube este notebook a GitHub *con sus salidas*. Por eso ninguna celda muestra caras ni el log del script de descarga: la inspección visual se guarda en `privado/` (output privado de Kaggle) y la descarga escribe en `privado/descarga.log`.

In [1]:
# Settings de Kaggle: Accelerator = GPU T4 x2 · Internet = On
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/Gredith/DeepLearning-G8-TA_Repository.git"
ROOT = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path.cwd().parents[1]
if Path("/kaggle").exists() and not ROOT.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src" / "kaggle_e2"))
print("ROOT =", ROOT)

ROOT = /kaggle/working/repo


In [2]:
!pip install -q --no-deps facenet-pytorch   # --no-deps: evita que reinstale otra versión de torch
import json, shutil, pandas as pd, torch
import data as D, preprocess as P, kaggle_io as K
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FFPP = Path("/tmp/ffpp")                 # videos: disco temporal, se borran tras recortar cada categoría
CROPS = Path("/tmp/crops_build/crops")    # recortes: se empaquetan en /kaggle/working/crops.tar
K.PRIVADO.mkdir(parents=True, exist_ok=True)
N_TRAIN, N_VAL, N_TEST, N_FRAMES, SEED = 100, 30, None, 16, 42

## 1. Qué videos se necesitan (splits oficiales, partición por video)

In [3]:
tr = D.subset_pairs(D.load_pairs("results/splits", "train"), N_TRAIN, SEED)
va = D.subset_pairs(D.load_pairs("results/splits", "val"), N_VAL, SEED)
te = D.subset_pairs(D.load_pairs("results/splits", "test"), N_TEST, SEED)
need, split_of = {}, {}
for split, pairs, cats in [("train", tr, ["Deepfakes"]), ("val", va, ["Deepfakes"]), ("test", te, D.METHODS)]:
    reals, fakes = D.video_ids(pairs)
    need.setdefault(D.REAL, set()).update(reals)
    split_of.update({v: split for v in reals + fakes})
    for c in cats: need.setdefault(c, set()).update(fakes)
need = {k: sorted(v) for k, v in need.items()}
# chequeo anti-fuga: ninguna identidad en dos splits
ids = {s: {v for p in ps for v in p} for s, ps in [("train", tr), ("val", va), ("test", te)]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
{k: len(v) for k, v in need.items()}

{'original': 400,
 'Deepfakes': 400,
 'Face2Face': 140,
 'FaceSwap': 140,
 'NeuralTextures': 140}

## 2. Descarga y recorte, una categoría a la vez
El script oficial de descarga llega con la aprobación del acceso. **No se versiona ni se enlaza** (términos de FF++): súbanlo a Kaggle como Dataset privado (*Add Input → Upload*); la celda lo encuentra sola.

Por cada categoría: descarga los 1000 videos c23 → recorta solo los necesarios (MTCNN, caja ×1.3, 16 frames equiespaciados) → **borra los videos**. Así el disco nunca guarda más de una categoría de videos. Si la sesión se corta, volver a correr: los videos ya recortados se saltan.

In [4]:
import subprocess
def _es_script_ffpp(p):
    n = p.name.lower()
    return n.endswith(".py") and "faceforensics" in n and "download" in n
SCRIPT = next((p for p in Path("/kaggle/input").rglob("*.py") if _es_script_ffpp(p)), None) if Path("/kaggle/input").exists() else None
assert SCRIPT, "No se encontró el script de FF++: súbanlo como Kaggle Dataset privado (Add Input → Upload)"
print("Script de descarga encontrado en los inputs privados.")   # sin ruta ni URL: esta salida va a GitHub

LOG = K.PRIVADO / "descarga.log"
rows = []
for cat in [D.REAL, *D.METHODS]:
    with open(LOG, "a") as log:   # `echo |` acepta los términos de uso que pide el script
        r = subprocess.run(f'echo "" | python "{SCRIPT}" "{FFPP}" -d {cat} -c c23 -t videos --server EU2',
                           shell=True, stdout=log, stderr=subprocess.STDOUT)
    assert r.returncode == 0, f"Falló la descarga de {cat}: revisar privado/descarga.log en el output"
    vdir = P.video_path(FFPP, cat, "x").parent
    n_vid = len(list(vdir.glob("*.mp4")))
    rows += P.process_all(FFPP, CROPS, {cat: need[cat]}, n_frames=N_FRAMES, device=DEVICE, progress=False)
    shutil.rmtree(vdir, ignore_errors=True)
    print(f"{cat:15s} videos descargados={n_vid:4d}  recortados={len(need[cat]):4d}  "
          f"recortes={sum(x['recortes'] for x in rows if x['categoria'] == cat)}")

Script de descarga encontrado en los inputs privados.
original        videos descargados=1000  recortados= 400  recortes=6400
Deepfakes       videos descargados=1000  recortados= 400  recortes=6394
Face2Face       videos descargados=1000  recortados= 140  recortes=2240
FaceSwap        videos descargados=1000  recortados= 140  recortes=2240
NeuralTextures  videos descargados=1000  recortados= 140  recortes=2240


## 3. Manifiesto, conteos y auditoría de descartes

In [5]:
man = pd.DataFrame(rows)
man["split"] = man.video.map(split_of)
man.to_csv(CROPS / "manifest.csv", index=False)
resumen = man.groupby("categoria").agg(videos=("video", "count"), sin_video=("existe_video", lambda s: (~s).sum()),
                                      recortes=("recortes", "sum"), esperados=("video", lambda s: len(s) * N_FRAMES),
                                      videos_sin_cara=("recortes", lambda s: (s == 0).sum()),
                                      frames_sin_cara=("frames_sin_cara", "sum"), frames_no_leidos=("frames_no_leidos", "sum"))
print(resumen)
incompletos = man[man.recortes < N_FRAMES]
print(f"\nVideos con menos de {N_FRAMES} recortes: {len(incompletos)}")
print(incompletos[["categoria", "video", "split", "recortes", "frames_sin_cara", "frames_no_leidos", "recortes_vacios"]].to_string(index=False))

                videos  sin_video  recortes  esperados  videos_sin_cara  \
categoria                                                                 
Deepfakes          400          0      6394       6400                0   
Face2Face          140          0      2240       2240                0   
FaceSwap           140          0      2240       2240                0   
NeuralTextures     140          0      2240       2240                0   
original           400          0      6400       6400                0   

                frames_sin_cara  frames_no_leidos  
categoria                                          
Deepfakes                     6                 0  
Face2Face                     0                 0  
FaceSwap                      0                 0  
NeuralTextures                0                 0  
original                      0                 0  

Videos con menos de 16 recortes: 3
categoria   video split  recortes  frames_sin_cara  frames_no_leidos  reco

In [6]:
# Inspección visual: real vs. cada manipulación del mismo target → solo a privado/, no se muestra
import matplotlib.pyplot as plt
from PIL import Image
a, b = te[0]
paths = [CROPS / D.REAL / a] + [CROPS / m / f"{a}_{b}" for m in D.METHODS]
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, p in zip(axes, paths):
    fs = sorted(p.glob("*.png"))
    if fs: ax.imshow(Image.open(fs[len(fs) // 2]))
    ax.set_title(p.parent.name, loc="left"); ax.axis("off")
K.save_private(fig, "ejemplo_recortes.png")

figura privada guardada (no se muestra por licencia FF++): privado/ejemplo_recortes.png


PosixPath('/kaggle/working/privado/ejemplo_recortes.png')

## 4. Empaquetar el output
Deja en `/kaggle/working` solo `crops.tar`, `manifest.csv`, `crops_meta.json` y `privado/`. Se borra la copia del repo para que el output sea liviano y fácil de convertir en Dataset.

In [7]:
import os
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
meta = {"commit": commit, "seed": SEED, "n_frames": N_FRAMES, "pares": {"train": N_TRAIN, "val": N_VAL, "test": len(te)},
        "recortes_por_categoria": {k: int(v) for k, v in man.groupby("categoria").recortes.sum().items()},
        "videos_incompletos": int(len(incompletos)), "detector": "MTCNN (facenet-pytorch), caja x1.3", "compresion": "c23"}
tar = K.pack_crops(CROPS, K.WORK)
shutil.copy(CROPS / "manifest.csv", K.WORK / "manifest.csv")
(K.WORK / "crops_meta.json").write_text(json.dumps(meta, indent=1))
os.chdir(K.WORK)
if K.KAGGLE:
    shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
    shutil.rmtree(CROPS.parent, ignore_errors=True)
print(f"{tar.name}: {tar.stat().st_size / 1e9:.2f} GB")
print(sorted(p.name for p in K.WORK.iterdir()))

crops.tar: 2.39 GB
['__notebook__.ipynb', 'crops.tar', 'crops_meta.json', 'manifest.csv', 'privado']


## 5. (Plan B) Crear el Dataset por API
Solo si *New Dataset* desde el output vuelve a fallar. Requiere un **Kaggle Secret** `KAGGLE_KEY` (Add-ons → Secrets) del integrante dueño del Dataset. Cambien `CREAR_POR_API = True` y vuelvan a guardar la versión. El Dataset se crea **privado** (es el valor por defecto de `kaggle datasets create`).

In [8]:
CREAR_POR_API = False
KAGGLE_USER = "ronaldosegura"   # dueño del Dataset
if CREAR_POR_API:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = KAGGLE_USER, UserSecretsClient().get_secret("KAGGLE_KEY")
    up = Path("/tmp/ds_upload"); up.mkdir(exist_ok=True)
    for f in ["crops.tar", "manifest.csv", "crops_meta.json"]:
        shutil.copy(K.WORK / f, up / f)
    (up / "dataset-metadata.json").write_text(json.dumps({"title": "ffpp-crops-g8", "id": f"{KAGGLE_USER}/ffpp-crops-g8",
                                                          "licenses": [{"name": "unknown"}]}))
    r = subprocess.run(["kaggle", "datasets", "create", "-p", str(up), "--dir-mode", "skip", "-q"], capture_output=True, text=True)
    print(r.returncode, r.stdout[-500:], r.stderr[-500:])
    shutil.rmtree(up)